[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rudrite/kernels/blob/main/labs/stage-1/lab-1.3-fused-softmax-layernorm.ipynb)

# LAB·1.3 · Fused softmax and LayerNorm

**Hardware:** correctness anywhere; the beats-XLA measurement needs a TPU runtime.

Elementwise chains are memory-bound, so the whole game is touching HBM once. Both kernels here load a block of rows, do all the math while the data sits in VMEM, and write once. That is what fusion means when you write it yourself.

In [ ]:
import time
import numpy as np
import jax
import jax.numpy as jnp
from jax.experimental import pallas as pl

print(jax.__version__, jax.devices())
ON_TPU = jax.devices()[0].platform == "tpu"
INTERP = not ON_TPU  # interpret mode anywhere; compiled kernels on a real TPU

def check(name, got, want, tol=2e-2):
    err = float(jnp.abs(got.astype(jnp.float32) - want.astype(jnp.float32)).max())
    status = "ok" if err <= tol else "FAIL"
    print(f"{name}: max err {err:.3e} [{status}]")
    assert err <= tol, name


In [ ]:
def softmax_kernel(x_ref, o_ref):
    x = x_ref[...].astype(jnp.float32)
    m = jnp.max(x, axis=-1, keepdims=True)
    e = jnp.exp(x - m)
    o_ref[...] = (e / jnp.sum(e, axis=-1, keepdims=True)).astype(o_ref.dtype)

def softmax(x, rows_per_block=8):
    n, d = x.shape
    return pl.pallas_call(
        softmax_kernel,
        grid=(n // rows_per_block,),
        in_specs=[pl.BlockSpec((rows_per_block, d), lambda i: (i, 0))],
        out_specs=pl.BlockSpec((rows_per_block, d), lambda i: (i, 0)),
        out_shape=jax.ShapeDtypeStruct(x.shape, x.dtype),
        interpret=INTERP,
    )(x)

x = jax.random.normal(jax.random.key(0), (256, 512), jnp.float32)
check("softmax", softmax(x), jax.nn.softmax(x, axis=-1), tol=1e-5)

One detail worth staring at: the row axis is blocked, the feature axis is not. The reduction runs entirely inside one block, which is exactly why this kernel is easy. Stage 3 is what happens when the reduction axis no longer fits.

In [ ]:
def layernorm_kernel(x_ref, g_ref, b_ref, o_ref):
    x = x_ref[...].astype(jnp.float32)
    mu = jnp.mean(x, axis=-1, keepdims=True)
    var = jnp.mean((x - mu) ** 2, axis=-1, keepdims=True)
    y = (x - mu) * jax.lax.rsqrt(var + 1e-6)
    o_ref[...] = (y * g_ref[...] + b_ref[...]).astype(o_ref.dtype)

def layernorm(x, g, b, rows_per_block=8):
    n, d = x.shape
    return pl.pallas_call(
        layernorm_kernel,
        grid=(n // rows_per_block,),
        in_specs=[
            pl.BlockSpec((rows_per_block, d), lambda i: (i, 0)),
            pl.BlockSpec((1, d), lambda i: (0, 0)),
            pl.BlockSpec((1, d), lambda i: (0, 0)),
        ],
        out_specs=pl.BlockSpec((rows_per_block, d), lambda i: (i, 0)),
        out_shape=jax.ShapeDtypeStruct(x.shape, x.dtype),
        interpret=INTERP,
    )(x, g.reshape(1, -1), b.reshape(1, -1))

g = jnp.ones(512); b = jnp.zeros(512)
ref = (x - x.mean(-1, keepdims=True)) * jax.lax.rsqrt(x.var(-1, keepdims=True) + 1e-6)
check("layernorm", layernorm(x, g, b), ref, tol=1e-4)

## Measure (TPU runtime)

Gate criterion: fused softmax beats the *unfused* XLA chain (`exp`, `max`, `sum` as separate jitted calls) at rows of 32k+. Also compare against `jax.nn.softmax` under one `jit`, and note what you find: XLA fuses that chain too, and matching it is the honest bar.

In [ ]:
if ON_TPU:
    big = jax.random.normal(jax.random.key(0), (32768, 512), jnp.bfloat16)
    def unfused(x):
        m = jax.jit(lambda v: jnp.max(v, -1, keepdims=True))(x)
        e = jax.jit(jnp.exp)(x - m)
        return e / jax.jit(lambda v: jnp.sum(v, -1, keepdims=True))(e)
    def bench(fn, *args, reps=20):
        fn(*args).block_until_ready()
        ts = []
        for _ in range(reps):
            t0 = time.perf_counter(); fn(*args).block_until_ready(); ts.append(time.perf_counter() - t0)
        return float(np.median(ts)) * 1e6
    print(f"unfused chain: {bench(unfused, big):8.0f} us")
    print(f"xla fused    : {bench(jax.jit(lambda v: jax.nn.softmax(v, -1)), big):8.0f} us")
    print(f"pallas fused : {bench(jax.jit(lambda v: softmax(v, 64)), big):8.0f} us  chip={jax.devices()[0].device_kind}")
else:
    print("Timing needs a TPU runtime.")